# **Predicting House Prices with Linear Regression**

---








## **Import Core Libraries**
#### We import the tools required for data manipulation, building the Linear Regression model, measuring its performance, and creating interactive graphs.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

## **Load and Inspect the Dataset**
#### We load the `Housing.csv` file into a Pandas DataFrame and view its metadata to check for missing records and inspect data types.

In [ ]:
# Load the dataset
df = pd.read_csv('Housing.csv')

# Display the first few rows
print("--- First 5 Rows of Dataset ---")
print(df.head())

# Display dataset details (missing values and data types)
print("\n--- Dataset Summary ---")
print(df.info())

--- First 5 Rows of Dataset ---
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  

--- Dataset Summary ---
<class 'pandas.core.frame.DataFrame'>
Rang

## **Data Preprocessing and Categorical Encoding**
#### Machine learning models require completely numerical inputs. We map **binary text (`yes`/`no`) into `1`/`0`** and apply **One-Hot Encoding** to structural status features.

In [ ]:
# Identify binary columns that need mapping to numerical flags
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

# Map yes to 1 and no to 0
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

# Apply One-Hot Encoding to the multi-level categorical column 'furnishingstatus'
# drop_first=True helps avoid multicollinearity (the dummy variable trap)
df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True)

# Verify processed columns
print("Processed Columns:")
print(df.columns.tolist())

Processed Columns:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished']


## **Feature Selection & Train-Test Splitting**
#### We isolate our features ($X$) from the target variable `price` ($y$), then split them using an **80/20 train-test split ratio**.

In [ ]:
# Separate features (X) and target (y)
X = df.drop('price', axis=1)
y = df['price']

# Split into 80% training and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set configuration: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Testing set configuration: {X_test.shape[0]} samples, {X_test.shape[1]} features")

Training set configuration: 436 samples, 13 features
Testing set configuration: 109 samples, 13 features


## **Train the Linear Regression Model**
#### We initialize the **Scikit-Learn Linear Regression algorithm** and train it using our split training subsets.

In [ ]:
# Initialize the linear regression model
model = LinearRegression()

# Train the model using the training datasets
model.fit(X_train, y_train)

print("Model successfully trained.")
print(f"Intercept: {model.intercept_:.2f}")

Model successfully trained.
Intercept: 260032.36


## **Make Predictions & Calculate Evaluation Metrics**
#### We predict prices using the test data and calculate **Mean Squared Error (MSE)** and the **R-squared ($R^2$) Score** to measure model accuracy.

In [ ]:
# Predict house prices for the test feature matrix
y_pred = model.predict(X_test)

# Calculate quantitative metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Evaluation Metrics ---")
print(f"Mean Squared Error (MSE): {mse:,.2f}")
print(f"R-squared (R2) Score  : {r2:.4f}")

--- Model Evaluation Metrics ---
Mean Squared Error (MSE): 1,754,318,687,330.66
R-squared (R2) Score  : 0.6529


## **Interactive Graph — Actual vs. Predicted Prices**
#### **Hover your mouse cursor over the points** to view live metric tooltips showing actual price, predicted value, area square footage, bedrooms, and bathrooms.

In [ ]:
# Build a consolidated DataFrame representing test observations for rich hover context
test_results = pd.DataFrame({
    'Actual_Price': y_test,
    'Predicted_Price': y_pred,
    'Area': X_test['area'],
    'Bedrooms': X_test['bedrooms'],
    'Bathrooms': X_test['bathrooms']
})

# Initialize Plotly Figure
fig = go.Figure()

# Add scatter plot tracking predicted values against actual targets
fig.add_trace(go.Scatter(
    x=test_results['Actual_Price'],
    y=test_results['Predicted_Price'],
    mode='markers',
    marker=dict(color='royalblue', size=10, opacity=0.75, line=dict(color='white', width=1)),
    name='House Data Point',
    hovertemplate=(
        '<b>Property Prediction Details:</b><br>' +
        'Actual Price: $%{x:,.0f}<br>' +
        'Predicted Price: $%{y:,.0f}<br>' +
        'Area Size: %{customdata[0]} sq ft<br>' +
        'Bedrooms Count: %{customdata[1]}<br>' +
        'Bathrooms Count: %{customdata[2]}<extra></extra>'
    ),
    customdata=test_results[['Area', 'Bedrooms', 'Bathrooms']].values
))

# Generate a reference perfect prediction line (y = x)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='firebrick', width=2, dash='dash'),
    name='Perfect Fit Line (y = x)'
))

# Configure visual layouts and titles
fig.update_layout(
    title='Actual vs. Predicted House Prices (Interactive)',
    xaxis_title='Actual Price ($)',
    yaxis_title='Predicted Price ($)',
    hovermode='closest',
    template='plotly_white',
    width=900,
    height=600
)

# Display graph inside your cell environment
fig.show()

## **Static Image Backup Plot (Seaborn)**
#### An alternative, production-ready static chart using Matplotlib/Seaborn saved directly to your local workspace.

In [ ]:
# Configure plot styling
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7, color='blue', edgecolor='w', s=80, ax=ax)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)

ax.set_title('Actual vs Predicted House Prices', fontsize=16, pad=15)
ax.set_xlabel('Actual Prices ($)', fontsize=12)
ax.set_ylabel('Predicted Prices ($)', fontsize=12)

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=300)
plt.close()
print("Static image successfully saved as 'actual_vs_predicted.png'.")


Static image successfully saved as 'actual_vs_predicted.png'.


## **Interactive Graph — Residuals Analysis Plot**
#### **Hover over the orange error points** to analyze model errors. Residuals ($\text{Actual} - \text{Predicted}$) should ideally scatter randomly around the $0$ Horizontal Line.

In [ ]:
# Calculate residuals
residuals = y_test - y_pred

# Create the Residuals Plot
fig_res = go.Figure()

# Add residual scatter points
fig_res.add_trace(go.Scatter(
    x=y_pred,
    y=residuals,
    mode='markers',
    marker=dict(color='darkorange', size=9, opacity=0.75, line=dict(color='white', width=1)),
    name='Residuals',
    hovertemplate=(
        '<b>Residual Details:</b><br>' +
        'Predicted Price: $%{x:,.0f}<br>' +
        'Error (Residual): $%{y:,.0f}<extra></extra>'
    )
))

# Add a horizontal zero reference line
fig_res.add_trace(go.Scatter(
    x=[y_pred.min(), y_pred.max()],
    y=[0, 0],
    mode='lines',
    line=dict(color='black', width=2, dash='dot'),
    name='Zero Error Line'
))

fig_res.update_layout(
    title='Residuals Plot (Predicted vs. Error)',
    xaxis_title='Predicted House Price ($)',
    yaxis_title='Residual / Error ($)',
    hovermode='closest',
    template='plotly_white',
    width=900,
    height=500
)

fig_res.show()

## **Interactive Graph — Distribution of Errors (Residual Histogram)**
#### Evaluates whether our model is systematically biased. **Hover over individual bars** to observe how many properties fall into specific error price brackets.

In [ ]:
# Create Histogram of Residuals
fig_hist = go.Figure()

fig_hist.add_trace(go.Histogram(
    x=residuals,
    nbinsx=20,
    marker_color='mediumseagreen',
    opacity=0.8,
    name='Error Distribution',
    hovertemplate='<b>Error Range:</b> %{x}<br><b>Count of Houses:</b> %{y}<extra></extra>'
))

fig_hist.update_layout(
    title='Distribution of Prediction Errors (Residuals)',
    xaxis_title='Prediction Error / Residual Value ($)',
    yaxis_title='Count of Properties',
    template='plotly_white',
    width=900,
    height=500
)

fig_hist.show()

## **Interactive Graph — Isolated Feature Trend Analysis (Area Size vs. Price)**
#### Plots the independent impact of continuous feature variables on valuation. **Hover to explore** the physical property area sizes alongside the calculated linear regression prediction path line.

In [ ]:
# Create data framework sorted by area for clean line visualization
area_axis = X_test['area']
sort_idx = np.argsort(area_axis)

fig_feature = go.Figure()

# Plot actual test samples
fig_feature.add_trace(go.Scatter(
    x=X_test['area'],
    y=y_test,
    mode='markers',
    marker=dict(color='purple', size=9, opacity=0.7),
    name='Actual Properties',
    hovertemplate='<b>Property Details:</b><br>Area: %{x} sq ft<br>Actual Price: $%{y:,.0f}<extra></extra>'
))

# Plot predicted trajectory
fig_feature.add_trace(go.Scatter(
    x=area_axis.iloc[sort_idx],
    y=y_pred[sort_idx],
    mode='lines+markers',
    line=dict(color='crimson', width=2),
    name='Predicted Trend',
    hovertemplate='<b>Predicted Trend Profile:</b><br>Area: %{x} sq ft<br>Predicted Price: $%{y:,.0f}<extra></extra>'
))

fig_feature.update_layout(
    title='House Area vs. Price (Actual Data vs. Model Trend Line)',
    xaxis_title='Area Size (Square Feet)',
    yaxis_title='House Price ($)',
    hovermode='closest',
    template='plotly_white',
    width=900,
    height=500
)

fig_feature.show()